## WP013 — Lineup continuity: does an unusual starting XI predict what the model misses?

See `README.md`. **P1 is pre-declared:** the opponent's defence continuity vs the side's residual goals (`goals − model λ`); hypothesised slope negative; a hit needs a 95% CI entirely below zero and a negative slope in both halves of the data. Everything else is exploratory. No sampling here.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from football_model.evaluation import market as mk
from football_model.features.continuity_features import build_continuity_table, UNITS

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP009 = REPO / 'work_products' / 'wp009_lineup_xg_validation'
N_BOOT = 5000

def show(df, digits=4):
    print(df.round(digits).to_string(index=False))

with open(WP001 / 'cv_shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
df_cv, windows = shared['df_cv'], shared['windows']
pmd = pd.read_pickle(WP009 / 'player_match_data.pkl')
table = build_continuity_table(pmd)
print(len(table), 'team-matches;', table['continuity_all'].notna().mean().round(3), 'have a value (rest are NaN by design)')
print(table[[f'continuity_{u}' for u in UNITS]].describe().loc[['count', 'mean', 'std', 'min', 'max']].round(3))

4560 team-matches; 0.868 have a value (rest are NaN by design)
       continuity_defence  continuity_midfield  continuity_attack  \
count            3960.000             3960.000           3958.000   
mean                0.714                0.658              0.641   
std                 0.140                0.138              0.265   
min                 0.060                0.040              0.000   
max                 1.000                1.000              1.000   

       continuity_all  
count        3960.000  
mean            0.679  
std             0.101  
min             0.109  
max             0.961  


### Held-out sides

One row per team per held-out match: the model's λ, the actual goals, and the continuity of that team (`own_*`) and of its opponent (`opp_*`), joined on `(team, date)`. The alignment between the checkpoint's stored predictions and the fixtures is the same as WP003/WP011, and is asserted against `df_cv`'s own goal columns.

In [2]:
def held_out_sides(ckpt):
    df_sorted = df_cv.sort_values('datetime').reset_index(drop=True)
    preds = ckpt['cv_match_predictions']
    rows = []
    for w in sorted({m['window'] for m in preds}):
        win = windows[w - 1]
        sel = df_sorted[(df_sorted['is_home'] == 1) & (df_sorted['round'] >= win['test_start']) & (df_sorted['round'] <= win['test_end'])]
        wp = [m for m in preds if m['window'] == w]
        assert len(sel) == len(wp), (w, len(sel), len(wp))
        for (_, r), m in zip(sel.iterrows(), wp):
            assert m['goals_home'] == r['goals'] and m['goals_away'] == r['goals_against'], 'fixture/prediction misaligned'
            date = pd.Timestamp(r['datetime']).normalize()
            key = f"{date.date()}_{r['team_long']}"
            rows.append({'match': key, 'window': w, 'date': date, 'team': r['team_long'], 'opp': r['opp_team_long'],
                         'goals': m['goals_home'], 'lam': m['lambda_home']})
            rows.append({'match': key, 'window': w, 'date': date, 'team': r['opp_team_long'], 'opp': r['team_long'],
                         'goals': m['goals_away'], 'lam': m['lambda_away']})
    s = pd.DataFrame(rows)
    s['resid'] = s['goals'] - s['lam']
    cont = table.rename(columns={c: c for c in table.columns})
    own = cont.add_prefix('own_').rename(columns={'own_team': 'team', 'own_date': 'date'})
    opp = cont.add_prefix('opp_').rename(columns={'opp_team': 'opp', 'opp_date': 'date'})
    s = s.merge(own, on=['team', 'date'], how='left').merge(opp, on=['opp', 'date'], how='left')
    return s

def slopes(s, features):
    """Goals-per-SD slope of residual goals on each feature: OLS with intercept
    (global centring), bootstrap CI clustered by match, and the slope in each
    half of the data (split at the median match date)."""
    first, second = mk.half_masks(s['date'])
    out = []
    for f in features:
        v = s[f].notna().to_numpy()
        d = s.loc[v]
        x = ((d[f] - d[f].mean()) / d[f].std()).to_numpy()
        r = (d['resid'] - d['resid'].mean()).to_numpy()
        g = d['match'].to_numpy()
        num = pd.Series(x * r).groupby(g).sum()
        den = pd.Series(x * x).groupby(g).sum()
        slope, lo, hi = mk.ratio_bootstrap(num.to_numpy(), den.to_numpy(), N_BOOT)
        h = []
        for m in (first[v], second[v]):
            xs, rs = x[m], r[m]
            h.append(float((xs * rs).sum() / (xs * xs).sum()))
        out.append({'feature': f, 'n_rows': int(v.sum()), 'slope': slope, 'lo': lo, 'hi': hi, 'half_1': h[0], 'half_2': h[1]})
    return pd.DataFrame(out)

with open(WP001 / 'cv_checkpoint.pkl', 'rb') as f:
    ckpt_base = pickle.load(f)
sides = held_out_sides(ckpt_base)
print(len(sides), 'side-rows,', sides['match'].nunique(), 'matches; mean residual', sides['resid'].mean().round(4), ' sd', sides['resid'].std().round(3))
print('rows with a valid opp_defence continuity:', sides['opp_continuity_defence'].notna().sum())

802 side-rows, 401 matches; mean residual 0.048  sd 1.197
rows with a valid opp_defence continuity: 702


### Slopes: goals per standard deviation of continuity

Each row regresses the side's residual goals on one standardised continuity score. Expected signs if the idea is right: a side's **own** midfield/attack/whole-XI continuity positive (settled attackers score more than the model expected); the **opponent's** defence continuity **negative** (that is P1). Rows with no expected sign are there as controls: if everything is "significant", the feature is picking up something generic.

In [3]:
FEATURES = [f'{side}_continuity_{u}' for side in ('opp', 'own') for u in ('defence', 'midfield', 'attack', 'all')]
res = slopes(sides, FEATURES)
show(res)

                feature  n_rows   slope      lo     hi  half_1  half_2
 opp_continuity_defence     702 -0.0769 -0.1683 0.0127 -0.0693 -0.0833
opp_continuity_midfield     702 -0.0033 -0.0869 0.0866  0.0036 -0.0110
  opp_continuity_attack     702 -0.0613 -0.1483 0.0210 -0.0626 -0.0603
     opp_continuity_all     702 -0.0792 -0.1683 0.0094 -0.0661 -0.0919
 own_continuity_defence     702 -0.0158 -0.1043 0.0761  0.0042 -0.0325
own_continuity_midfield     702  0.0546 -0.0323 0.1420  0.0390  0.0717
  own_continuity_attack     702 -0.0117 -0.1013 0.0747 -0.0342  0.0068
     own_continuity_all     702  0.0189 -0.0676 0.1061  0.0172  0.0206


## P1 verdict

Pre-declared: `opp_continuity_defence`, slope negative, 95% CI entirely below zero, both halves negative.

In [4]:
p1 = res[res['feature'] == 'opp_continuity_defence'].iloc[0]
hit = (p1['hi'] < 0) and (p1['half_1'] < 0) and (p1['half_2'] < 0)
print(f"P1 opp_continuity_defence: slope {p1['slope']:+.4f} goals/SD, 95% CI [{p1['lo']:+.4f}, {p1['hi']:+.4f}], "
      f"halves {p1['half_1']:+.4f} / {p1['half_2']:+.4f}, n_rows={int(p1['n_rows'])}  ->  {'HIT' if hit else 'no hit'}")

# detectable size at this n: about 2 standard errors
se = (p1['hi'] - p1['lo']) / (2 * 1.96)
print(f'approximate standard error {se:.4f} goals/SD, so effects below about {2 * se:.3f} goals/SD are not detectable here')

P1 opp_continuity_defence: slope -0.0769 goals/SD, 95% CI [-0.1683, +0.0127], halves -0.0693 / -0.0833, n_rows=702  ->  no hit
approximate standard error 0.0462 goals/SD, so effects below about 0.092 goals/SD are not detectable here


### Robustness: same P1 test on the residuals of `lineup_loose_combo`

WP009's full 35-window `lineup_loose_combo` predictions already include the attacking lineup term, so if defence continuity were only proxying the same information the slope would shrink. Exploratory.

In [5]:
with open(WP009 / 'cv_checkpoint_full_lineup_loose_combo.pkl', 'rb') as f:
    ckpt_lineup = pickle.load(f)
sides_l = held_out_sides(ckpt_lineup)
show(slopes(sides_l, ['opp_continuity_defence', 'own_continuity_attack', 'own_continuity_all']))

               feature  n_rows   slope      lo     hi  half_1  half_2
opp_continuity_defence     702 -0.0741 -0.1654 0.0150 -0.0661 -0.0808
 own_continuity_attack     702 -0.0157 -0.1055 0.0702 -0.0372  0.0021
    own_continuity_all     702  0.0179 -0.0676 0.1042  0.0147  0.0209
